# Disneyland RAG System Demo

This notebook demonstrates the RAG system for answering questions about Disneyland visitor reviews.

## Setup

Load environment, instantiate embeddings and LLM.

In [1]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from rag.config import DATA_PATH, EMBEDDING_MODEL_NAME, LLM_MODEL_NAME, LLM_TEMPERATURE, LLM_MAX_TOKENS, LLM_TIMEOUT
from rag.embeddings import SentenceTransformerEmbeddings
from rag.ingest import load_reviews
from rag.vectorstore import get_or_build_collection
from rag.chain import ask
from langchain_litellm import ChatLiteLLM

print(f"Data path: {DATA_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"LLM model: {LLM_MODEL_NAME}")

/home/syaramionak/Projects/rag-system-poc/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
16:29:36 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
16:29:37 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Data path: /home/syaramionak/Projects/rag-system-poc/data/DisneylandReviews.csv
Embedding model: all-MiniLM-L6-v2
LLM model: litellm_proxy/openrouter/openai/gpt-4.1-mini


## Load and Embed Data

This cell loads reviews from CSV and embeds them into ChromaDB.
On first run, this takes 3-5 minutes. Subsequent runs load from disk instantly.

In [2]:
# Load reviews
print("Loading reviews from CSV...")
documents = load_reviews(DATA_PATH)
print(f"Loaded {len(documents)} reviews")

# Initialize embeddings
print(f"\nInitializing embeddings with {EMBEDDING_MODEL_NAME}...")
embeddings = SentenceTransformerEmbeddings()

# Build or load ChromaDB collection
print("Building/loading ChromaDB collection...")
collection = get_or_build_collection(documents, embeddings)
print(f"Collection size: {collection.count()} documents")

Loading reviews from CSV...
  (Skipped 20 duplicate review IDs)
Loaded 42636 reviews

Initializing embeddings with all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13462.97it/s]


Building/loading ChromaDB collection...

📚 EMBEDDINGS LOADED FROM CACHE (instant)
   Location: /home/syaramionak/Projects/rag-system-poc/chroma_db
   Documents: 42,636
   Status: Ready to use

Collection size: 42636 documents


## Initialize LLM

Create a ChatLiteLLM instance pointing to the LiteLLM proxy.

In [3]:
import os

proxy_url = os.getenv("LITELLM_PROXY_URL", "https://litellm.gke-prod.linnovate.net")
api_key = os.getenv("LITELLM_MASTER_KEY")

print(f"Proxy URL: {proxy_url}")
print(f"API Key: {'***' if api_key else 'NOT SET'}")

llm = ChatLiteLLM(
    model=LLM_MODEL_NAME,
    api_base=proxy_url,
    api_key=api_key,
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
    timeout=LLM_TIMEOUT,
)
print("\nLLM initialized successfully")
print(f"Temperature: {LLM_TEMPERATURE}, Max tokens: {LLM_MAX_TOKENS}")

Proxy URL: https://litellm.gke-prod.linnovate.net
API Key: ***

LLM initialized successfully
Temperature: 0.7, Max tokens: 1024


## Example 1: Australia visitors at HongKong

Question: "What do visitors from Australia say about Disneyland in HongKong?"

In [4]:
question_1 = "What do visitors from Australia say about Disneyland in HongKong?"
print(f"Question: {question_1}")
print("\n" + "="*60)

answer_1 = ask(question_1, collection, embeddings, llm, auto_extract_filters=True, n_results=10)
print("Answer:")
print(answer_1)

Question: What do visitors from Australia say about Disneyland in HongKong?

Answer:
Visitors from Australia generally have positive things to say about Disneyland in Hong Kong. Many highlight the park as a magical and joyful place that brings happiness around every corner. They appreciate that it is close to Australia, making it a convenient option compared to the Disneyland parks in the USA. The park is described as clean, well-maintained, and having friendly staff. Several visitors mention that the park has grown over the years with new lands and attractions that are great for children.

However, some Australian visitors note a few downsides, such as long queues and occasional queue-cutting by other visitors. A few also point out that the park is smaller and has fewer attractions compared to the Disneyland parks in the US. The food is considered average and somewhat overpriced. Additionally, some rides with audio are only in Cantonese, which can be a letdown for those who do not und

## Example 2: Spring season visits

Question: "Is spring a good time to visit Disneyland?"

In [5]:
question_2 = "Is spring a good time to visit Disneyland?"
print(f"Question: {question_2}")
print("\n" + "="*60)

answer_2 = ask(question_2, collection, embeddings, llm, auto_extract_filters=True, n_results=10)
print("Answer:")
print(answer_2)

Question: Is spring a good time to visit Disneyland?

Answer:
Based on the visitor reviews, spring can be a mixed time to visit Disneyland. Some visitors mention that spring is a good time to enjoy the park, with pleasant weather and enjoyable events like the electric light parade and fireworks. However, others caution that spring break, which occurs during spring, is very crowded and can result in long lines and packed areas.

In summary:
- Spring (outside of spring break) can be a nice time to visit with good weather and enjoyable experiences.
- Spring break itself is crowded and may be less enjoyable due to large crowds and longer wait times.

Using Fast Pass options and apps for wait times can help make a spring visit more efficient and enjoyable.


## Example 3: California in June

Question: "Is Disneyland California usually crowded in June?"

In [6]:
question_3 = "Is Disneyland California usually crowded in June?"
print(f"Question: {question_3}")
print("\n" + "="*60)

answer_3 = ask(question_3, collection, embeddings, llm, auto_extract_filters=True, n_results=30)
print("Answer:")
print(answer_3)

Question: Is Disneyland California usually crowded in June?

Answer:
Based on the visitor reviews provided, Disneyland California is usually quite crowded in June. Multiple reviewers mentioned large crowds, with some describing them as "huge and at times overbearing" and others noting that the park is "simply too small for the number of folks who are there on an average day." However, it's also mentioned that going early when the park opens can help avoid long lines and crowds, as lines tend to be shorter in the morning. Overall, June appears to be a busy time at Disneyland California.


## Example 4: Staff friendliness in Paris

Question: "Is the staff in Paris friendly?"

In [7]:
question_4 = "Is the staff in Paris friendly?"
print(f"Question: {question_4}")
print("\n" + "="*60)

answer_4 = ask(question_4, collection, embeddings, llm, auto_extract_filters=True, n_results=30)
print("Answer:")
print(answer_4)

Question: Is the staff in Paris friendly?

Answer:
The reviews about staff friendliness at Disneyland Paris are mixed:

- Several visitors from the United Kingdom and other countries report that the staff are unfriendly, rude, or unhelpful. For example, one review from 2011 states, "The staff are rude and unhelpful," and another from 2012 mentions "miserable, bolshy staff." Some also mention that staff made little effort to assist visitors.

- On the other hand, some reviews praise the staff for being friendly, helpful, and professional. For instance, a 2013 review from the UK says the staff were "much better friendlier and more helpful," and a 2015 review from the US states, "The staff were amazing they seriously couldn't help more." Another 2013 review highlights that staff were "very helpful" especially for special assistance needs.

- One review suggests that American management presence might have temporarily improved staff friendliness.

In summary, experiences with staff friendl

## Debug: Inspect Retrieved Documents

For a given question, see what documents are retrieved before they go to the LLM.

In [8]:
from rag.filter_parser import extract_filters
from rag.retriever import retrieve

debug_question = "Is Disneyland California usually crowded in June?"
print(f"Debug question: {debug_question}")

# Extract filters
filters = extract_filters(debug_question, llm)
print(f"\nExtracted filters: {filters}")

# Retrieve documents with all filters
retrieved_docs = retrieve(
    debug_question,
    collection,
    embeddings,
    n_results=30,
    branch=filters.get("branch"),
    reviewer_location=filters.get("reviewer_location"),
    season=filters.get("season"),
    min_rating=filters.get("min_rating"),
    year_month=filters.get("year_month"),
    prefer_recent=filters.get("prefer_recent"),
)
print(f"\nRetrieved {len(retrieved_docs)} documents:")
print("\n" + "-"*60 + "\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"  Metadata: {doc.metadata}")
    print(f"  Text: {doc.page_content[:200]}...")
    print()

Debug question: Is Disneyland California usually crowded in June?

Extracted filters: {'branch': 'Disneyland_California', 'reviewer_location': None, 'season': 'summer', 'min_rating': None, 'year_month': '6', 'prefer_recent': False}

Retrieved 9 documents:

------------------------------------------------------------

Document 1:
  Metadata: {'branch': 'Disneyland_California', 'season': 'summer', 'rating': 5, 'year_month': '2012-6', 'review_id': '133651989', 'reviewer_location': 'United States'}
  Text: You can never go wrong with Disneyland. If you ever do travel there in June though beware of them closing early for Grad Night. I am not a fan of Grad Night. They close the Park early in order for the...

Document 2:
  Metadata: {'branch': 'Disneyland_California', 'reviewer_location': 'United States', 'season': 'summer', 'rating': 3, 'review_id': '279373773', 'year_month': '2015-6'}
  Text: Listen, we LOVE Disneyland.. we live in Utah and have annual passes.. we drive all the way just fo